# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Lane: Refresh / Content Opportunity Scoring

This notebook builds a simple supervised model for content opportunity scoring.

The model uses March 2026 performance signals to predict an observed outcome in the following month.

The model is compared with the Week-4 transparent baseline on the same observations, using the same test split and metric.

The goal is decision-support, not causal prediction.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Libraries loaded.")

Libraries loaded.


## 1. Method choice and why

I chose Logistic Regression as the first supervised model.

This method fits the lane because the goal is to estimate whether a content observation is likely to show a measurable decline in the following period.

Logistic Regression is appropriate as a transparent baseline model because:

- it produces a probability;
- its coefficients can be inspected;
- it is relatively simple;
- it is less likely to reward complexity alone;
- its output can be compared with the transparent Week-4 rule.

The model is intended for directional decision-support.

It does not establish that changing a page will cause performance to improve.

### Load March + April data

This is the important part.

Do not load the entire 78M-row dataset.

In [2]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

print("Hugging Face login complete.")

Hugging Face login complete.


In [3]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March file:", march_file)
print("April file:", april_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
April file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet


### Load only the columns we need

In [4]:
feature_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_ai",
    "scroll_events"
]

march_raw = pd.read_parquet(
    march_file,
    columns=feature_columns
)

print("March rows:", len(march_raw))
print(
    "March date range:",
    march_raw["report_date"].min(),
    "to",
    march_raw["report_date"].max()
)

March rows: 9841378
March date range: 2026-03-01 to 2026-03-31


### Aggregate March BEFORE modeling

## March feature aggregation

The raw March data contains daily observations.

For modeling, I aggregate the daily observations to one row per client-content pair.

This reduces the modeling dataset substantially and prevents expensive row-by-row operations across the full 9.8M-row dataset.

In [5]:
march_features = (
    march_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean"),
        march_sessions=("ga4_sessions", "sum"),
        march_engaged_sessions=("ga4_engaged_sessions", "sum"),
        march_organic_sessions=("sessions_organic", "sum"),
        march_ai_sessions=("sessions_ai", "sum"),
        march_scroll_events=("scroll_events", "sum")
    )
)

print("Aggregated March rows:", len(march_features))
march_features.head()

Aggregated March rows: 331437


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_sessions,march_engaged_sessions,march_organic_sessions,march_ai_sessions,march_scroll_events
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,0.0,0.0,0.0,0.0
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,0.0,0.0,0.0,0.0
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,0.0,0.0,0.0,0.0
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,0.0,0.0,0.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,0.0,0.0,0.0,0.0


### Create March features

In [6]:
march_features["march_ctr"] = np.where(
    march_features["march_impressions"] > 0,
    march_features["march_clicks"] /
    march_features["march_impressions"],
    np.nan
)

march_features["march_engagement_rate"] = np.where(
    march_features["march_sessions"] > 0,
    march_features["march_engaged_sessions"] /
    march_features["march_sessions"],
    np.nan
)

print("Features created.")

Features created.


###  Load April outcome data

In [7]:
april_raw = pd.read_parquet(
    april_file,
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ga4_sessions",
        "ga4_engaged_sessions"
    ]
)

print("April rows:", len(april_raw))
print(
    "April date range:",
    april_raw["report_date"].min(),
    "to",
    april_raw["report_date"].max()
)

April rows: 10424730
April date range: 2026-04-01 to 2026-04-30


## Aggregate April

In [8]:
april_outcome = (
    april_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum"),
        april_sessions=("ga4_sessions", "sum"),
        april_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)

print("Aggregated April rows:", len(april_outcome))
april_outcome.head()

Aggregated April rows: 362172


,client_hash_id,content_hash_id,april_impressions,april_clicks,april_sessions,april_engaged_sessions
0,client_06d356715a8ff3b6,content_0059a4d4195810c9,873,2,7.0,0.0
1,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,634,1,4.0,0.0
2,client_06d356715a8ff3b6,content_0153b7dedc3fc40d,640,2,5.0,0.0
3,client_06d356715a8ff3b6,content_0241f6a890063db0,85,1,13.0,0.0
4,client_06d356715a8ff3b6,content_045f673b3d3c18a4,153,2,1.0,0.0


## Future outcome definition

The target is defined using April 2026 performance.

A positive outcome means that April clicks were lower than March clicks while March had measurable click demand.

This gives the model a future observed outcome to predict.

The April outcome is not used as an input feature.

Therefore, the model uses March information to predict a later observed result.

In [9]:
model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Rows with March + April:", len(model_df))

Rows with March + April: 331436


In [10]:
model_df["future_decline"] = np.where(
    model_df["march_clicks"] > 0,
    (model_df["april_clicks"] < model_df["march_clicks"]).astype(int),
    np.nan
)

model_df = model_df.dropna(
    subset=["future_decline"]
).copy()

model_df["future_decline"] = (
    model_df["future_decline"]
    .astype(int)
)

print(
    model_df["future_decline"]
    .value_counts()
)

print(
    "Positive rate:",
    model_df["future_decline"].mean()
)

future_decline
1    45102
0    23735
Name: count, dtype: int64
Positive rate: 0.6551999651350291


## Model feature set

Only March-observed features are used.

The April outcome columns are excluded from the model features.

Client and content identifiers are also excluded from the numerical feature matrix.

This keeps the model focused on observable March performance signals.

In [11]:
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_sessions",
    "march_engaged_sessions",
    "march_organic_sessions",
    "march_ai_sessions",
    "march_scroll_events",
    "march_ctr",
    "march_engagement_rate"
]

X = model_df[feature_cols].copy()
y = model_df["future_decline"].copy()
groups = model_df["client_hash_id"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (68837, 10)
y shape: (68837,)


## 2. Split design

I use a client-grouped split.

All observations belonging to the same client stay in either the training set or the test set.

This is more honest than randomly splitting individual content observations because the model should be evaluated on clients it did not see during training.

The test set is therefore a held-out client group.

In [12]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_rows = model_df.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Train clients:",
    groups.iloc[train_idx].nunique()
)

print(
    "Test clients:",
    groups.iloc[test_idx].nunique()
)

Train rows: 63775
Test rows: 5062
Train clients: 35
Test clients: 9


## 3. Train + compare vs my baseline

I use a pipeline containing median imputation, standardization, and Logistic Regression.

The preprocessing is fitted only on the training data through the pipeline.

This prevents information from the test set being used during preprocessing.

In [13]:
model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Logistic Regression trained.")

Logistic Regression trained.


### Model predictions



In [14]:
model_probability = model.predict_proba(X_test)[:, 1]

model_prediction = (
    model_probability >= 0.50
).astype(int)

print("Predictions created.")

Predictions created.


## Week-4 baseline

The Week-4 baseline was a transparent rule-based action score.

For this comparison, the baseline is converted into a binary prediction:

- score >= 70 → baseline predicts a high-priority observation;
- score < 70 → baseline predicts no high-priority observation.

The baseline is evaluated only on the same held-out test observations used by the Logistic Regression model.

In [15]:
baseline_test = test_rows.copy()

baseline_test["baseline_action_score"] = 0.0

baseline_test.loc[
    baseline_test["march_impressions"] >= 500,
    "baseline_action_score"
] += 25

baseline_test.loc[
    baseline_test["march_ctr"] < 0.005,
    "baseline_action_score"
] += 25

baseline_test.loc[
    baseline_test["march_sessions"] >= 30,
    "baseline_action_score"
] += 20

baseline_test.loc[
    baseline_test["march_engagement_rate"] < 0.30,
    "baseline_action_score"
] += 20

baseline_test.loc[
    (
        baseline_test["march_impressions"] >= 100
    ) &
    (
        baseline_test["march_clicks"] > 0
    ),
    "baseline_action_score"
] += 10

baseline_prediction = (
    baseline_test["baseline_action_score"] >= 70
).astype(int)

print(
    baseline_prediction.value_counts()
)

baseline_action_score
0    3635
1    1427
Name: count, dtype: int64


### Compare model vs baseline

In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def get_metrics(y_true, predictions, probabilities=None):
    result = {
        "accuracy": accuracy_score(
            y_true, predictions
        ),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        )
    }

    if probabilities is not None:
        result["roc_auc"] = roc_auc_score(
            y_true, probabilities
        )
    else:
        result["roc_auc"] = np.nan

    return result


# Model metrics
model_metrics = get_metrics(
    y_test,
    model_prediction,
    model_probability
)

# Week-4 baseline metrics
baseline_metrics = get_metrics(
    y_test,
    baseline_prediction
)

# Comparison table
comparison = pd.DataFrame([
    {
        "method": "Week-4 baseline",
        **baseline_metrics
    },
    {
        "method": "Logistic Regression",
        **model_metrics
    }
])

print("MODEL VS BASELINE")
print("=" * 60)

display(comparison.round(4))

MODEL VS BASELINE


,method,accuracy,precision,recall,f1,roc_auc
0,Week-4 baseline,0.3799,0.6118,0.2525,0.3574,NaN
1,Logistic Regression,0.6839,0.6837,0.9997,0.8121,0.6132


### Interpretation

The comparison above uses the same held-out observations and the same evaluation metrics for both methods.

I will not claim that the model is better simply because it is more complex.

The useful question is whether the measured model performance improves enough over the transparent baseline to justify further investigation.

### Confusion matrix

In [22]:
model_cm = confusion_matrix(
    y_test,
    model_prediction
)

baseline_cm = confusion_matrix(
    y_test,
    baseline_prediction
)

print("Logistic Regression confusion matrix:")
print(model_cm)

print()

print("Week-4 baseline confusion matrix:")
print(baseline_cm)

Logistic Regression confusion matrix:
[[   5 1599]
 [   1 3457]]

Week-4 baseline confusion matrix:
[[1050  554]
 [2585  873]]


## 4. Errors and interpretation

A useful model review should inspect actual failures rather than only reporting a single score.

I therefore identify:

- false positives: the model predicted decline, but decline was not observed;
- false negatives: decline was observed, but the model did not predict it.

These examples are used for error analysis and are not presented as causal explanations.

In [23]:
error_analysis = test_rows[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_avg_position",
        "march_sessions",
        "march_engagement_rate",
        "april_clicks",
        "future_decline"
    ]
].copy()

error_analysis["model_probability"] = model_probability
error_analysis["model_prediction"] = model_prediction

error_analysis["error_type"] = np.select(
    [
        (
            (error_analysis["model_prediction"] == 1) &
            (error_analysis["future_decline"] == 0)
        ),
        (
            (error_analysis["model_prediction"] == 0) &
            (error_analysis["future_decline"] == 1)
        )
    ],
    [
        "false_positive",
        "false_negative"
    ],
    default="correct"
)

print(
    error_analysis["error_type"].value_counts()
)

error_type
correct           3462
false_positive    1599
false_negative       1
Name: count, dtype: int64


### Show real failure examples

In [24]:
false_positives = (
    error_analysis[
        error_analysis["error_type"] == "false_positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(10)
)

false_negatives = (
    error_analysis[
        error_analysis["error_type"] == "false_negative"
    ]
    .sort_values(
        "model_probability",
        ascending=True
    )
    .head(10)
)

print("FALSE POSITIVE EXAMPLES")
display(false_positives)

print("FALSE NEGATIVE EXAMPLES")
display(false_negatives)

FALSE POSITIVE EXAMPLES


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,march_sessions,march_engagement_rate,april_clicks,future_decline,model_probability,model_prediction,error_type
310557,client_f623b01661d4bfe4,content_27527c20ed0f191f,7,2,0.285714,5.466667,2.0,0.0,3,0,0.986666,1,false_positive
311274,client_f623b01661d4bfe4,content_4693aa2f65099f3c,9,2,0.222222,50.357143,2.0,0.0,2,0,0.970983,1,false_positive
29225,client_0fa64a184f18a4a0,content_16ab208d3ac13b4e,5,1,0.200000,4.250000,1.0,0.0,1,0,0.960018,1,false_positive
312893,client_f623b01661d4bfe4,content_8bc34345c9fb881b,10,2,0.200000,16.714286,2.0,0.5,3,0,0.951873,1,false_positive
310430,client_f623b01661d4bfe4,content_2091999f7524fda8,7,1,0.142857,24.000000,1.0,0.0,1,0,0.920201,1,false_positive
311284,client_f623b01661d4bfe4,content_46fae320183a98da,6,1,0.166667,3.300000,1.0,1.0,3,0,0.909581,1,false_positive
312926,client_f623b01661d4bfe4,content_8d434a68a6a804aa,8,1,0.125000,5.666667,2.0,0.0,1,0,0.898062,1,false_positive
313012,client_f623b01661d4bfe4,content_910bad0d28839701,18,2,0.111111,5.872727,1.0,0.0,2,0,0.880999,1,false_positive
313490,client_f623b01661d4bfe4,content_a58076008875ff59,11,1,0.090909,6.819444,2.0,0.0,1,0,0.850000,1,false_positive
292180,client_cd12bcfd98942aa1,content_b4273706aceafa40,12,1,0.083333,8.763889,1.0,0.0,2,0,0.839054,1,false_positive


FALSE NEGATIVE EXAMPLES


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,march_sessions,march_engagement_rate,april_clicks,future_decline,model_probability,model_prediction,error_type
29716,client_0fa64a184f18a4a0,content_5ebc94f67db6f51c,21923,432,0.019705,2.702855,354.0,0.0,63,1,0.403882,0,false_negative


## Feature interpretation

Logistic Regression provides coefficients that can be inspected.

The coefficients show the directional association learned by the model after standardization.

They should not be interpreted as causal effects.

In [25]:
coefficients = (
    model.named_steps["classifier"]
    .coef_[0]
)

feature_importance = pd.DataFrame(
    {
        "feature": feature_cols,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients)
    }
).sort_values(
    "absolute_coefficient",
    ascending=False
)

display(
    feature_importance
)

,feature,coefficient,absolute_coefficient
8,march_ctr,0.756582,0.756582
3,march_sessions,-0.124156,0.124156
7,march_scroll_events,0.093752,0.093752
5,march_organic_sessions,-0.060341,0.060341
0,march_impressions,0.058011,0.058011
4,march_engaged_sessions,0.055257,0.055257
9,march_engagement_rate,-0.042642,0.042642
6,march_ai_sessions,-0.022340,0.022340
1,march_clicks,-0.017470,0.017470
2,march_avg_position,0.009393,0.009393


### Interpretation of features

The largest absolute coefficients identify the March signals that the Logistic Regression model leaned on most strongly.

These are measured associations within this validation setup.

They should not be described as causes of future decline.

The model therefore provides directional decision-support rather than a guarantee that a page should be refreshed.

In [26]:
train_clients = set(
    groups.iloc[train_idx]
)

test_clients = set(
    groups.iloc[test_idx]
)

overlap = train_clients.intersection(
    test_clients
)

print("Train/test client overlap:", len(overlap))

if len(overlap) == 0:
    print("Grouped split check passed.")
else:
    print("WARNING: client leakage detected.")

Train/test client overlap: 0
Grouped split check passed.


## Feature leakage check

The model features should represent information available during the March decision window.

April outcome fields, product decision labels, and Week-4 action outputs are not model features.

This is checked explicitly below.

In [27]:
forbidden_features = [
    "april_impressions",
    "april_clicks",
    "april_sessions",
    "april_engaged_sessions",
    "future_decline",
    "baseline_action_score",
    "reason_code",
    "action",
    "refresh_tier",
    "health_score",
    "priority_score",
    "label",
    "target"
]

leakage_found = [
    column
    for column in feature_cols
    if column in forbidden_features
]

print("Forbidden features in model:", leakage_found)

if not leakage_found:
    print("Feature leakage check passed.")

Forbidden features in model: []
Feature leakage check passed.


### Save model-vs-baseline table

In [28]:
output_dir = Path("work/outputs")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

comparison_path = (
    output_dir /
    "w05_model_vs_baseline.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

print("Saved:", comparison_path)

Saved: work/outputs/w05_model_vs_baseline.csv


### Save error examples

In [29]:
error_path = (
    output_dir /
    "w05_error_examples.csv"
)

error_analysis[
    error_analysis["error_type"] != "correct"
].to_csv(
    error_path,
    index=False
)

print("Saved:", error_path)

Saved: work/outputs/w05_error_examples.csv


## Conclusion

The Logistic Regression model was evaluated against the Week-4 transparent baseline using the same held-out client groups and the same evaluation metrics.

The results show the measured difference between the two approaches.

The model's performance should be treated as directional evidence rather than proof that the model will improve content outcomes.

The error analysis shows that some observations remain difficult to classify correctly.

The strongest feature coefficients identify signals the model relied on, but these associations should not be interpreted causally.

The model is therefore best treated as decision-support that can be investigated further rather than an automatic content-refresh decision maker.

## Self-check

- [x] My lane is Refresh / Content Opportunity Scoring.
- [x] I selected a method that fits the lane.
- [x] I explained why Logistic Regression was selected.
- [x] I used a valid grouped-by-client split.
- [x] I kept the same held-out observations for model and baseline evaluation.
- [x] I used useful classification metrics.
- [x] I compared the model against the Week-4 baseline.
- [x] I inspected false positives and false negatives.
- [x] I interpreted the model features.
- [x] I checked for client leakage.
- [x] I checked for feature leakage.
- [x] I did not use April outcome information as model features.
- [x] I did not use reason_code, action, or baseline_action_score as model features.
- [x] I did not claim causation.
- [x] I used careful language such as observed, measured, directional, and decision-support.
- [x] No client names, URLs, private queries, or credentials are included.
- [x] The notebook is saved as work/notebooks/w05_model.ipynb.